# 03 — Normalization Experiments

Measures whether the normalization in `business_entity_resolution.normalization`
actually improves exact-match rate, candidate retrieval, and positive/negative
pair separation, using real training data and ground truth (not toy examples).

Sample size here is chosen to keep the notebook fast to re-execute; the same
code runs unchanged over the full 12.5M-row training set inside the CLI
(`python -m business_entity_resolution train`).

In [1]:

import sys, time
sys.path.insert(0, r"C:\Users\mudit\Documents\projects\ml_amazon\code\business_entity_resolution\src")
import pandas as pd
import numpy as np
from business_entity_resolution.data import loader
from business_entity_resolution.normalization.run_normalize import normalize_source
from business_entity_resolution import config

N = 300_000
s1 = loader.load_source(config.TRAIN_FILES["S1"], nrows=N)
s2 = loader.load_source(config.TRAIN_FILES["S2"], nrows=N)
s3 = loader.load_source(config.TRAIN_FILES["S3"], nrows=N)
gt = loader.load_ground_truth(config.TRAIN_GROUND_TRUTH, nrows=N)
gt_long = loader.explode_ground_truth(gt)
print(len(s1), len(s2), len(s3), len(gt_long))


300000 300000 300000 1038755


## Normalize all three sources and time it

In [2]:

t0 = time.time()
s1n = normalize_source(s1)
s2n = normalize_source(s2)
s3n = normalize_source(s3)
print(f"normalized {len(s1n)+len(s2n)+len(s3n)} rows in {time.time()-t0:.1f}s")


normalized 900000 rows in 61.4s


## Build the true-positive pair sample

We only have ground-truth rows whose `source1_entity_id`/matched ids happen to
fall inside these truncated N-row slices (file order differs between S1 and
ground truth, so this is a random subsample of true pairs, not the first N).

In [3]:

others = pd.concat([s2n, s3n], ignore_index=True).set_index("entity_id")
s1_idx = s1n.set_index("entity_id")

valid_gt = gt_long[gt_long["source1_entity_id"].isin(s1_idx.index) & gt_long["matched_entity_id"].isin(others.index)]
print("usable true positive pairs in this sample:", len(valid_gt))

pos = valid_gt.merge(s1_idx[[
    "business_name","name_normalized","name_compact","name_core","name_tokens","name_ngrams",
    "business_address","address_normalized","address_tokens","address_ngrams",
]], left_on="source1_entity_id", right_index=True, suffixes=("", "_s1"))
pos = pos.join(others[[
    "business_name","name_normalized","name_compact","name_core","name_tokens","name_ngrams",
    "business_address","address_normalized","address_tokens","address_ngrams",
]], on="matched_entity_id", rsuffix="_cand")
pos = pos.rename(columns={c: f"s1_{c}" for c in [
    "business_name","name_normalized","name_compact","name_core","name_tokens","name_ngrams",
    "business_address","address_normalized","address_tokens","address_ngrams",
]})
print(pos.shape)
pos.head(3)


usable true positive pairs in this sample: 8304


(8304, 22)


,source1_entity_id,matched_entity_id,s1_business_name,s1_name_normalized,s1_name_compact,s1_name_core,s1_name_tokens,s1_name_ngrams,s1_business_address,s1_address_normalized,...,business_name_cand,name_normalized_cand,name_compact_cand,name_core_cand,name_tokens_cand,name_ngrams_cand,business_address_cand,address_normalized_cand,address_tokens_cand,address_ngrams_cand
188,S1-244810289,S2-63598201,Celestial Memorial Trust,celestial memorial trust,celestial memorial trust,celestial memorial trust,"[celestial, memorial, trust]","[ cel, cele, eles, lest, esti, stia, tial, ial...","No. 35, Brentwood Apartments, Defence Colony, ...",no 35 brentwood apartments defence colony 2nd ...,...,Celestial Memorial,celestial memorial,celestial memorial,celestial memorial,"[celestial, memorial]","[ cel, cele, eles, lest, esti, stia, tial, ial...","35 , BRENTWOOD APARTMENTS, DEFENCE COLONY, 2ND...",35 brentwood apartments defence colony 2nd mai...,"[35, brentwood, apartments, defence, colony, 2...","[ 35 , 35 b, 5 br, bre, bren, rent, entw, ntw..."
314,S1-541270202,S2-657479988,"Sofie Greenman, O.D.",sofie greenman o d,d greenman o sofie,sofie greenman o d,"[sofie, greenman, o, d]","[ sof, sofi, ofie, fie , ie g, e gr, gre, gre...","356 Bell Factory Road, Huntsville, AL",356 bell factory rd huntsville al,...,"O.D., Sofie Greenman,",o d sofie greenman,d greenman o sofie,o d sofie greenman,"[o, d, sofie, greenman]","[ o d, o d , d s, d so, sof, sofi, ofie, fie...","356 BELL FACTORY ROAD, HUNTSVILLE, AL",356 bell factory rd huntsville al,"[356, bell, factory, rd, huntsville, al]","[ 356, 356 , 56 b, 6 be, bel, bell, ell , ll ..."
502,S1-996012175,S2-269681023,Ferguson Webster PLLC,ferguson webster pllc,ferguson pllc webster,ferguson webster pllc,"[ferguson, webster, pllc]","[ fer, ferg, ergu, rgus, guso, uson, son , on ...","880 June Terrace, Unit UNIT 270, Lake Zurich, IL",880 june ter unit unit 270 lake zurich il,...,Ferguson Webster,ferguson webster,ferguson webster,ferguson webster,"[ferguson, webster]","[ fer, ferg, ergu, rgus, guso, uson, son , on ...","880 JUNE TER, LAKE ZRUICH CDP, IL",880 june ter lake zruich cdp il,"[880, june, ter, lake, zruich, cdp, il]","[ 880, 880 , 80 j, 0 ju, jun, june, une , ne ..."


## Exact-match rate: raw text vs. normalized vs. compact (word-order invariant)

In [4]:

def rate(a, b):
    return float((a == b).mean())

print("NAME")
print("  raw exact match:        ", rate(pos["s1_business_name"], pos["business_name_cand"]))
print("  normalized exact match: ", rate(pos["s1_name_normalized"], pos["name_normalized_cand"]))
print("  compact exact match:    ", rate(pos["s1_name_compact"], pos["name_compact_cand"]))
print("  core (no suffix) match: ", rate(pos["s1_name_core"], pos["name_core_cand"]))

print("ADDRESS")
print("  raw exact match:        ", rate(pos["s1_business_address"], pos["business_address_cand"]))
print("  normalized exact match: ", rate(pos["s1_address_normalized"], pos["address_normalized_cand"]))


NAME
  raw exact match:         0.042509633911368014
  normalized exact match:  0.24157032755298652
  compact exact match:     0.29804913294797686
  core (no suffix) match:  0.4172687861271676
ADDRESS
  raw exact match:         0.02053931451612903
  normalized exact match:  0.12210982658959538


This directly mirrors the EDA's positive-pair analysis (raw exact ≈ a few
percent, normalization lifts it materially, compact/core lift it further by
absorbing word-order and suffix noise) — confirming normalization is worth
the engineering cost before we ever get to fuzzy similarity.

## Token / n-gram Jaccard: positive pairs vs. random (likely-negative) pairs

In [5]:

from business_entity_resolution.features._utils import jaccard

name_tok_jac_pos = jaccard(pos["s1_name_tokens"], pos["name_tokens_cand"])
name_ngram_jac_pos = jaccard(pos["s1_name_ngrams"], pos["name_ngrams_cand"])

rng = np.random.RandomState(0)
neg_cand_idx = rng.choice(len(others), size=len(pos), replace=True)
neg_cand = others.iloc[neg_cand_idx]
name_tok_jac_neg = jaccard(pos["s1_name_tokens"].reset_index(drop=True), neg_cand["name_tokens"].reset_index(drop=True))
name_ngram_jac_neg = jaccard(pos["s1_name_ngrams"].reset_index(drop=True), neg_cand["name_ngrams"].reset_index(drop=True))

print("mean name token jaccard   | positive pairs:", name_tok_jac_pos.mean(), " random pairs:", name_tok_jac_neg.mean())
print("mean name ngram jaccard   | positive pairs:", name_ngram_jac_pos.mean(), " random pairs:", name_ngram_jac_neg.mean())


mean name token jaccard   | positive pairs: 0.6352345972107533  random pairs: 0.02262917851790684
mean name ngram jaccard   | positive pairs: 0.6476917194629148  random pairs: 0.011540565637852948


## Takeaways

- Raw-string exact match recovers only a small fraction of true pairs; normalization
  (casefold, Unicode NFKC, punctuation/whitespace cleanup, suffix canonicalization)
  substantially increases exact-match yield, consistent with the EDA's own
  positive-pair analysis.
- Token/char-n-gram Jaccard cleanly separates true pairs from random pairs
  (positive-pair mean is many times the random-pair mean), which is why both
  are used as blocking/feature signals rather than raw string equality alone.
- Multiple representations (`normalized`, `compact`, `core`) are kept because
  they lift recall through *different* noise types (case/punctuation vs.
  word-order vs. suffix variation) — collapsing to one aggressive canonical
  form would blur that.
